# NB05 — Targeted AEDA 50% cho OFFENSIVE và HATE

NB05 kiểm định giả thuyết rút ra từ log NB04: augmentation 25% trên cả CLEAN làm tăng mạnh false positive, trong khi lợi ích chủ yếu nằm ở recall của lớp thiểu số. Variant mới chỉ augment **50% mẫu train đủ điều kiện của OFFENSIVE và HATE**, tuyệt đối không tạo mẫu CLEAN.

Mục tiêu không phải ép recall tăng bằng mọi giá, mà xem OFFENSIVE/HATE F1 và Macro-F1 có tăng mà CLEAN FPR vẫn được kiểm soát hay không. Notebook báo riêng precision/recall/F1 từng lớp và paired bootstrap 95% CI.

In [ ]:
# USER SETTINGS — phải là code cell đầu tiên, trước khi import torch/transformers
SEED = 42
MODEL_VARIANTS = ('original', 'aeda_vi_toxic50_v1')
AUGMENTATION_VERSION = 'aeda_vi_toxic50_v1'
AUGMENT_RATE = 0.5
AUGMENT_TARGET_LABELS = (1, 2)
AUGMENTATION_SEED = 20261022
MAX_AEDA_EDITS = 3
EPOCHS = 10.0
EVALUATE_TEST = True
MAX_LENGTH, MICRO_BATCH, GRAD_ACCUMULATION, EVAL_BATCH = 128, 8, 5, 16
LAST_N_LAYERS, CNN_DIM, CNN_KERNELS, FUSION_DIM = 4, 192, (3, 5, 7), 256
CNN_DROPOUT, HEAD_DROPOUT = 0.10, 0.15
ENCODER_LR, HEAD_LR, WEIGHT_DECAY, WARMUP_RATIO = 1.2e-5, 4e-5, 0.01, 0.10
CLASS_WEIGHT_POWER, FOCAL_GAMMA = 0.35, 0.5
AUX_TOXIC_WEIGHT, AUX_TARGET_WEIGHT = 0.20, 0.25
GATE_SCALE_LOGIT_INIT = -1.5
CALIBRATION_RADIUS, CALIBRATION_STEP = 1.0, 0.05
BOOTSTRAP_SAMPLES = 2000
TEST_INTERPRETATION = 'post_hoc_augmentation_ablation_after_nb03_test_observation'

import os, sys, subprocess
from datetime import datetime, timezone
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert MICRO_BATCH * GRAD_ACCUMULATION == 40
assert MODEL_VARIANTS[0] == 'original' and MODEL_VARIANTS[1] == AUGMENTATION_VERSION
assert 0.0 < AUGMENT_RATE <= 1.0 and set(AUGMENT_TARGET_LABELS) <= {0, 1, 2}
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'datasets', 'accelerate', 'sentencepiece', 'protobuf',
    'huggingface_hub', 'scikit-learn', 'pandas', 'pyarrow', 'matplotlib', 'safetensors'],
    check=True)
RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = Path('/kaggle/working') / f'optimized_gate_v4_toxic50_seed42_{RUN_TAG}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
print({'phase': 'setup', 'seed': SEED, 'model_variants': MODEL_VARIANTS,
       'augmentation_version': AUGMENTATION_VERSION, 'augmentation_rate': AUGMENT_RATE,
       'target_labels': AUGMENT_TARGET_LABELS, 'effective_batch': 40,
       'evaluate_test': EVALUATE_TEST, 'run_dir': str(RUN_DIR)})


In [ ]:
import gc, hashlib, json, math, random
from collections import Counter
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
from transformers import AutoTokenizer, set_seed

assert torch.cuda.is_available() and torch.cuda.device_count() == 1
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
random.seed(SEED); np.random.seed(SEED); set_seed(SEED)
try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    HF_ACCOUNT = HfApi(token=HF_TOKEN).whoami()['name']
except Exception as exc:
    raise RuntimeError('Enable a valid Kaggle Secret named HF_TOKEN.') from exc

manifest_paths = list(Path('/kaggle/input').rglob('vihsd_manifest.json'))
if len(manifest_paths) != 1:
    raise RuntimeError(f'Attach exactly one prepared ViHSD output: {manifest_paths}')
manifest_path = manifest_paths[0]
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if manifest.get('artifact') != 'vihsd_prepared_v1':
    raise ValueError('Unexpected prepared-data artifact.')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def canonical_hash(payload):
    encoded = json.dumps(payload, ensure_ascii=False, sort_keys=True,
        separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()

def load_verified_frame(split):
    spec = manifest['splits'][split]
    path = manifest_path.parent / spec['file']
    if sha256(path) != spec['sha256']:
        raise ValueError(f'{split} hash mismatch')
    frame = pd.read_parquet(path)
    if len(frame) != spec['rows'] or not set(frame.labels.unique()) <= {0, 1, 2}:
        raise ValueError(f'{split} content mismatch')
    return frame

MODEL_ID, MODEL_SHA = manifest['model_id'], manifest['model_revision']
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_SHA, token=HF_TOKEN)
tokenizer.padding_side = 'right'
content_special_ids = sorted({int(value) for value in (
    tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id,
    tokenizer.cls_token_id, tokenizer.sep_token_id) if value is not None})

def make_dataset(frame):
    dataset = Dataset.from_dict({
        'text': frame.text.fillna('').astype(str).tolist(),
        'labels': frame.labels.astype(int).tolist(),
    })
    return dataset.map(lambda batch: tokenizer(batch['text'], truncation=True,
        max_length=MAX_LENGTH), batched=True, remove_columns=['text'])

AEDA_PUNCTUATION = ('!', '?', '.', ',', ';', ':')

def insert_aeda_punctuation(text, rng):
    tokens = str(text).split()
    if not tokens:
        raise ValueError('AEDA requires at least one token.')
    edits = min(MAX_AEDA_EDITS, max(1, int(round(0.30 * len(tokens)))))
    augmented = list(tokens)
    for _ in range(edits):
        position = int(rng.integers(0, len(augmented) + 1))
        punctuation = AEDA_PUNCTUATION[int(rng.integers(0, len(AEDA_PUNCTUATION)))]
        augmented.insert(position, punctuation)
    return ' '.join(augmented), edits

def build_controlled_aeda(frame, rate, target_labels, seed):
    required = {'text', 'labels'}
    if not required <= set(frame.columns):
        raise ValueError(f'Missing train columns: {required - set(frame.columns)}')
    target_labels = tuple(sorted(set(int(label) for label in target_labels)))
    rng = np.random.default_rng(seed)
    original_texts = set(frame.text.fillna('').astype(str))
    reserved_texts = set(original_texts)
    records, report = [], {}
    for label in target_labels:
        eligible = []
        for index in frame.index[frame.labels.astype(int).eq(label)]:
            text = str(frame.at[index, 'text']).strip()
            if not text:
                continue
            token_count = len(tokenizer(text, add_special_tokens=True,
                truncation=False)['input_ids'])
            if token_count + MAX_AEDA_EDITS <= MAX_LENGTH:
                eligible.append(index)
        target_count = int(math.floor(rate * len(eligible)))
        accepted = 0
        for index in rng.permutation(eligible):
            if accepted >= target_count:
                break
            original = str(frame.at[index, 'text']).strip()
            augmented, edits = insert_aeda_punctuation(original, rng)
            if augmented in reserved_texts:
                continue
            encoded_length = len(tokenizer(augmented, add_special_tokens=True,
                truncation=False)['input_ids'])
            if encoded_length > MAX_LENGTH:
                continue
            record = {'source_index': str(index), 'text': augmented,
                'labels': int(label), 'edits': int(edits),
                'augmentation_version': AUGMENTATION_VERSION}
            records.append(record); reserved_texts.add(augmented); accepted += 1
        report[str(label)] = {'eligible': len(eligible), 'target': target_count,
            'accepted': accepted, 'acceptance_rate': accepted / max(len(eligible), 1)}
    augmented = pd.DataFrame(records)
    if augmented.empty:
        raise RuntimeError('No augmented rows were generated.')
    if not set(augmented.labels.astype(int).unique()) <= set(target_labels):
        raise AssertionError('Augmentation escaped the requested target labels.')
    combined = pd.concat([frame[['text', 'labels']], augmented[['text', 'labels']]],
        ignore_index=True)
    if len(combined) != len(frame) + len(augmented):
        raise AssertionError('Combined train row count mismatch.')
    artifact = {
        'artifact': 'controlled_aeda_train_only_v1',
        'augmentation_version': AUGMENTATION_VERSION,
        'augmentation_seed': int(seed), 'candidate_rate': float(rate),
        'target_labels': list(target_labels), 'max_length': MAX_LENGTH,
        'max_edits': MAX_AEDA_EDITS, 'source_train_sha256': manifest['splits']['train']['sha256'],
        'original_rows': len(frame), 'augmented_rows': len(augmented),
        'combined_rows': len(combined), 'per_class': report,
        'augmented_label_counts': {str(key): int(value) for key, value in
            Counter(augmented.labels.astype(int)).items()},
        'qc_status': 'automated_checks_only_exploratory',
    }
    artifact['augmentation_hash'] = canonical_hash(records)
    return combined, augmented, artifact

# Chỉ đọc train/validation ở giai đoạn xây corpus và huấn luyện.
train_frame = load_verified_frame('train')
validation_frame = load_verified_frame('validation')
augmented_train_frame, augmentation_rows, augmentation_manifest = build_controlled_aeda(
    train_frame, AUGMENT_RATE, AUGMENT_TARGET_LABELS, AUGMENTATION_SEED)
augmentation_rows.to_parquet(RUN_DIR / 'augmentation_rows.parquet', index=False)
(RUN_DIR / 'augmentation_manifest.json').write_text(json.dumps(
    augmentation_manifest, indent=2, ensure_ascii=False), encoding='utf-8')

train_frames = {'original': train_frame, AUGMENTATION_VERSION: augmented_train_frame}
train_datasets = {name: make_dataset(frame) for name, frame in train_frames.items()}
validation_ds = make_dataset(validation_frame)
train_counts = np.bincount(train_frame.labels.astype(int), minlength=3).astype(np.float64)
if np.any(train_counts == 0):
    raise ValueError('All three classes must occur in train.')
# Cố định weights theo train gốc để chỉ đo tác động của augmentation.
class_weights = train_counts ** (-CLASS_WEIGHT_POWER)
class_weights /= class_weights.mean()
toxic_count = train_counts[1] + train_counts[2]
toxic_pos_weight = float(np.sqrt(train_counts[0] / toxic_count))
print({'gpu': torch.cuda.get_device_name(0), 'hf_account': HF_ACCOUNT,
       'train_counts_original': train_counts.astype(int).tolist(),
       'train_rows_by_variant': {key: len(value) for key, value in train_frames.items()},
       'class_weights_from_original': class_weights.round(4).tolist(),
       'toxic_aux_pos_weight': round(toxic_pos_weight, 4),
       'augmentation_manifest': augmentation_manifest,
       'model_revision': MODEL_SHA, 'dataset_revision': manifest['dataset_revision']})


## Can thiệp duy nhất: phân bổ augmentation

Model, loss, seed, learning rates, batch và tổng optimizer updates giống NB04. Khác biệt duy nhất là candidate pool: `OFFENSIVE=50%`, `HATE=50%`, `CLEAN=0%`. Class weights vẫn lấy từ train gốc để tránh đồng thời thay hai cơ chế cân bằng lớp. Manifest lưu số eligible/target/accepted theo lớp và assertion chặn mọi augmented row mang nhãn CLEAN.

In [ ]:
from torch import nn
import torch.nn.functional as F
from transformers import AutoConfig, AutoModel, PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput

LABEL2ID = {'CLEAN': 0, 'OFFENSIVE': 1, 'HATE': 2}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}

class OptimizedGateConfig(PretrainedConfig):
    model_type, is_composition = 'visobert-optimized-gate-v4', True

    def __init__(self, encoder_config=None, encoder_model_id='uitnlp/visobert',
                 encoder_revision=None, last_n_layers=4, cnn_dim=192,
                 cnn_kernels=(3, 5, 7), fusion_dim=256, cnn_dropout=.1,
                 head_dropout=.15, class_weights=None, focal_gamma=.5,
                 aux_toxic_weight=.2, aux_target_weight=.25, toxic_pos_weight=1.,
                 gate_scale_logit_init=-1.5, content_special_ids=None,
                 num_labels=3, **kwargs):
        kernels = tuple(int(value) for value in cnn_kernels)
        if last_n_layers < 1 or any(value < 1 or value % 2 == 0 for value in kernels):
            raise ValueError('Invalid layer count or CNN kernels.')
        super().__init__(num_labels=num_labels, **kwargs)
        self.encoder_config = dict(encoder_config or {})
        self.encoder_model_id, self.encoder_revision = encoder_model_id, encoder_revision
        self.last_n_layers, self.cnn_dim = int(last_n_layers), int(cnn_dim)
        self.cnn_kernels, self.fusion_dim = list(kernels), int(fusion_dim)
        self.cnn_dropout, self.head_dropout = float(cnn_dropout), float(head_dropout)
        self.class_weights = list(class_weights or [])
        self.focal_gamma = float(focal_gamma)
        self.aux_toxic_weight = float(aux_toxic_weight)
        self.aux_target_weight = float(aux_target_weight)
        self.toxic_pos_weight = float(toxic_pos_weight)
        self.gate_scale_logit_init = float(gate_scale_logit_init)
        self.content_special_ids = list(content_special_ids or [])

class ResidualDsConv(nn.Module):
    def __init__(self, channels, kernel, dropout):
        super().__init__()
        self.depthwise = nn.Conv1d(channels, channels, kernel, padding=kernel // 2,
            groups=channels, bias=False)
        self.pointwise = nn.Conv1d(channels, channels, 1)
        self.dropout, self.norm = nn.Dropout(dropout), nn.LayerNorm(channels)

    def forward(self, tokens, mask):
        valid = mask.unsqueeze(-1).to(tokens.dtype)
        tokens = tokens * valid
        update = self.pointwise(F.gelu(
            self.depthwise(tokens.transpose(1, 2)))).transpose(1, 2)
        return self.norm(tokens + self.dropout(update)) * valid

class MultiScaleDsCnnPool(nn.Module):
    def __init__(self, hidden, channels, kernels, dropout):
        super().__init__()
        self.projection = nn.Linear(hidden, channels)
        self.norm = nn.LayerNorm(channels)
        self.branches = nn.ModuleList([ResidualDsConv(channels, kernel, dropout)
            for kernel in kernels])
        self.output_size = 2 * channels * len(kernels)

    def forward(self, hidden, mask):
        if hidden.shape[:2] != mask.shape or not bool(mask.any(dim=1).all()):
            raise ValueError('Invalid hidden/mask pair.')
        valid = mask.unsqueeze(-1).to(hidden.dtype)
        tokens = self.norm(F.gelu(self.projection(hidden))) * valid
        pooled = []
        for branch in self.branches:
            features = branch(tokens, mask)
            maximum = features.masked_fill(~mask.unsqueeze(-1),
                torch.finfo(features.dtype).min).amax(dim=1)
            mean = (features * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)
            pooled.extend((maximum, mean))
        return torch.cat(pooled, dim=-1)

def relation_features(left, right):
    return torch.cat([left, right, torch.abs(left - right), left * right], dim=-1)

def binary_head(input_dim, hidden_dim, dropout):
    return nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(),
        nn.LayerNorm(hidden_dim), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

class OptimizedGateViSoBERT(PreTrainedModel):
    config_class, base_model_prefix, main_input_name = OptimizedGateConfig, 'encoder', 'input_ids'
    accepts_loss_kwargs = False

    def __init__(self, config):
        super().__init__(config)
        payload = dict(config.encoder_config)
        model_type = payload.pop('model_type', None)
        if not model_type:
            raise ValueError('encoder_config.model_type is required')
        encoder_config = AutoConfig.for_model(model_type, **payload)
        encoder_config._attn_implementation = 'eager'
        if config.last_n_layers > int(encoder_config.num_hidden_layers):
            raise ValueError('last_n_layers exceeds encoder depth')
        hidden, dim = int(encoder_config.hidden_size), config.fusion_dim
        self.encoder = AutoModel.from_config(encoder_config, add_pooling_layer=False)
        self.layer_weights = nn.Parameter(torch.zeros(config.last_n_layers))
        self.global_projection = nn.Linear(2 * hidden, dim)
        self.global_norm = nn.LayerNorm(dim)
        self.local_pool = MultiScaleDsCnnPool(hidden, config.cnn_dim,
            config.cnn_kernels, config.cnn_dropout)
        self.local_projection = nn.Linear(self.local_pool.output_size, dim)
        self.local_norm = nn.LayerNorm(dim)
        self.gate_hidden = nn.Linear(4 * dim, dim)
        self.gate_output = nn.Linear(dim, dim)
        self.residual_scale_logit = nn.Parameter(
            torch.tensor(config.gate_scale_logit_init))
        self.fusion_norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(config.head_dropout)
        self.classifier = nn.Linear(dim, config.num_labels)
        self.toxic_head = binary_head(4 * dim, dim, config.head_dropout)
        self.target_head = binary_head(4 * dim, dim, config.head_dropout)
        self.register_buffer('class_weight_tensor',
            torch.tensor(config.class_weights, dtype=torch.float32), persistent=False)
        self.register_buffer('toxic_pos_weight_tensor',
            torch.tensor(config.toxic_pos_weight, dtype=torch.float32), persistent=False)
        self.post_init()
        nn.init.zeros_(self.gate_output.weight)
        nn.init.zeros_(self.gate_output.bias)

    def content_mask(self, input_ids, attention_mask):
        mask = attention_mask.bool().clone()
        for token_id in self.config.content_special_ids:
            mask &= input_ids.ne(int(token_id))
        empty = ~mask.any(dim=1)
        if empty.any():
            mask[empty] = attention_mask[empty].bool()
        return mask

    def compute_features(self, input_ids, attention_mask, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
            output_hidden_states=True, return_dict=True, **kwargs)
        selected = torch.stack(outputs.hidden_states[-self.config.last_n_layers:], dim=0)
        mixing = self.layer_weights.softmax(dim=0).view(-1, 1, 1, 1)
        hidden = (selected * mixing).sum(dim=0)
        mask = self.content_mask(input_ids, attention_mask)
        valid = mask.unsqueeze(-1).to(hidden.dtype)
        mean = (hidden * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)
        global_vector = self.global_norm(F.gelu(self.global_projection(
            torch.cat([hidden[:, 0], mean], dim=-1))))
        local_vector = self.local_norm(F.gelu(self.local_projection(
            self.local_pool(hidden, mask))))
        relations = relation_features(global_vector, local_vector)
        gate = torch.sigmoid(self.gate_output(F.gelu(self.gate_hidden(relations))))
        rho = torch.sigmoid(self.residual_scale_logit)
        fused = self.fusion_norm(global_vector + rho * gate *
            (local_vector - global_vector))
        return fused, global_vector, local_vector, gate, rho, outputs

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        if attention_mask is None:
            raise ValueError('attention_mask is required')
        kwargs.pop('num_items_in_batch', None)
        fused, global_vector, local_vector, _, _, outputs = self.compute_features(
            input_ids, attention_mask, **kwargs)
        logits = self.classifier(self.dropout(fused))
        loss = None
        if labels is not None:
            per_sample_ce = F.cross_entropy(logits, labels,
                weight=self.class_weight_tensor.to(logits.dtype), reduction='none')
            true_probability = logits.softmax(dim=-1).gather(
                1, labels[:, None]).squeeze(1)
            main_loss = (((1.0 - true_probability).clamp_min(0) **
                self.config.focal_gamma) * per_sample_ce).mean()
            relations = relation_features(global_vector, local_vector)
            toxic_logit = self.toxic_head(relations).squeeze(-1)
            target_logit = self.target_head(relations).squeeze(-1)
            toxic_truth = labels.ne(0).to(logits.dtype)
            toxic_loss = F.binary_cross_entropy_with_logits(toxic_logit, toxic_truth,
                pos_weight=self.toxic_pos_weight_tensor.to(logits.dtype))
            toxic_mask = labels.ne(0)
            if toxic_mask.any():
                target_truth = labels.eq(2).to(logits.dtype)
                target_loss = F.binary_cross_entropy_with_logits(
                    target_logit[toxic_mask], target_truth[toxic_mask])
            else:
                target_loss = logits.sum() * 0.0
            loss = (main_loss + self.config.aux_toxic_weight * toxic_loss +
                self.config.aux_target_weight * target_loss)
        return SequenceClassifierOutput(loss=loss, logits=logits)

    @torch.no_grad()
    def gate_diagnostics(self, input_ids, attention_mask, **kwargs):
        _, _, _, gate, rho, _ = self.compute_features(
            input_ids, attention_mask, **kwargs)
        return gate, rho, self.layer_weights.softmax(dim=0)

encoder_config = AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_SHA, token=HF_TOKEN)

def build_fresh_model():
    config = OptimizedGateConfig(encoder_config=encoder_config.to_dict(),
        encoder_model_id=MODEL_ID, encoder_revision=MODEL_SHA,
        last_n_layers=LAST_N_LAYERS, cnn_dim=CNN_DIM, cnn_kernels=CNN_KERNELS,
        fusion_dim=FUSION_DIM, cnn_dropout=CNN_DROPOUT, head_dropout=HEAD_DROPOUT,
        class_weights=class_weights.tolist(), focal_gamma=FOCAL_GAMMA,
        aux_toxic_weight=AUX_TOXIC_WEIGHT, aux_target_weight=AUX_TARGET_WEIGHT,
        toxic_pos_weight=toxic_pos_weight, gate_scale_logit_init=GATE_SCALE_LOGIT_INIT,
        content_special_ids=content_special_ids, num_labels=3,
        label2id=LABEL2ID, id2label=ID2LABEL)
    model = OptimizedGateViSoBERT(config)
    pretrained = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_SHA, token=HF_TOKEN,
        add_pooling_layer=False, attn_implementation='eager')
    copied = model.encoder.load_state_dict(pretrained.state_dict(), strict=True)
    del pretrained
    if copied.missing_keys or copied.unexpected_keys:
        raise RuntimeError(copied)
    return model, {'total': sum(value.numel() for value in model.parameters()),
        'trainable': sum(value.numel() for value in model.parameters() if value.requires_grad)}

In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix,
    precision_recall_fscore_support)

def metrics_from_predictions(labels, predicted):
    labels, predicted = np.asarray(labels), np.asarray(predicted)
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, predicted, labels=[0, 1, 2], zero_division=0)
    matrix = confusion_matrix(labels, predicted, labels=[0, 1, 2])
    return {'accuracy': float(accuracy_score(labels, predicted)),
        'macro_f1': float(f1.mean()),
        'weighted_f1': float(np.average(f1, weights=support)),
        'per_class_precision': {ID2LABEL[i]: float(precision[i]) for i in range(3)},
        'per_class_recall': {ID2LABEL[i]: float(recall[i]) for i in range(3)},
        'per_class_f1': {ID2LABEL[i]: float(f1[i]) for i in range(3)},
        'support': {ID2LABEL[i]: int(support[i]) for i in range(3)},
        'confusion_matrix': matrix.tolist()}

def metrics_from_logits(labels, logits):
    return metrics_from_predictions(labels, np.asarray(logits).argmax(axis=-1))

def compute_metrics(prediction):
    values = metrics_from_logits(prediction.label_ids, prediction.predictions)
    return {key: values[key] for key in ('accuracy', 'macro_f1', 'weighted_f1')}

def fit_validation_biases(logits, labels, radius=1.0, step=0.05):
    values, truth = np.asarray(logits, dtype=np.float64), np.asarray(labels)
    candidates = np.arange(-radius, radius + step / 2, step)
    raw = metrics_from_logits(truth, values)['macro_f1']
    best_key, best_biases = (raw, 0.0, 0.0, 0.0), np.zeros(3)
    for offensive_bias in candidates:
        for hate_bias in candidates:
            biases = np.array([0.0, offensive_bias, hate_bias])
            score = metrics_from_logits(truth, values + biases)['macro_f1']
            key = (score, -float(biases @ biases), -abs(offensive_bias), -abs(hate_bias))
            if key > best_key:
                best_key, best_biases = key, biases
    return {'biases': best_biases.tolist(), 'raw_macro_f1': raw,
        'calibrated_macro_f1': float(best_key[0]), 'source': 'validation_only',
        'radius': float(radius), 'step': float(step)}

def bootstrap_macro_f1_ci(labels, predicted, seed, seed_offset=0):
    truth, guess = np.asarray(labels), np.asarray(predicted)
    groups = [np.flatnonzero(truth == label) for label in range(3)]
    rng, scores = np.random.default_rng(seed + 3000 + seed_offset), []
    for _ in range(BOOTSTRAP_SAMPLES):
        indices = np.concatenate([rng.choice(group, len(group), replace=True)
            for group in groups])
        scores.append(metrics_from_predictions(
            truth[indices], guess[indices])['macro_f1'])
    return [float(value) for value in np.quantile(scores, [0.025, 0.975])]

def clean_false_positive_rate(metrics):
    matrix = metrics['confusion_matrix']
    return float((matrix[0][1] + matrix[0][2]) / metrics['support']['CLEAN'])

def flatten_seed_metrics(seed, raw_metrics, calibrated_metrics, diagnostics=None):
    row = {'seed': int(seed)}
    for prefix, values in (('raw', raw_metrics), ('calibrated', calibrated_metrics)):
        row[f'{prefix}_accuracy'] = values['accuracy']
        row[f'{prefix}_macro_f1'] = values['macro_f1']
        row[f'{prefix}_weighted_f1'] = values['weighted_f1']
        row[f'{prefix}_clean_fpr'] = clean_false_positive_rate(values)
        for label_name, column_name in (
                ('CLEAN', 'clean'), ('OFFENSIVE', 'offensive'), ('HATE', 'hate')):
            row[f'{prefix}_{column_name}_precision'] = values[
                'per_class_precision'][label_name]
            row[f'{prefix}_{column_name}_recall'] = values[
                'per_class_recall'][label_name]
            row[f'{prefix}_{column_name}_f1'] = values['per_class_f1'][label_name]
    if diagnostics is not None:
        row['gate_rho'] = diagnostics['rho']
        row['gate_mean'] = diagnostics['gate_mean']
        row['gate_std'] = diagnostics['gate_std']
        row['effective_local_coefficient_mean'] = diagnostics[
            'effective_local_coefficient_mean']
    return row

def summarize_seed_frame(frame):
    if len(frame) < 2:
        raise ValueError('At least two seeds are required for sample SD.')
    metric_columns = [column for column in frame.columns if column != 'seed']
    summary = frame[metric_columns].agg(['mean', 'std']).T.reset_index()
    summary.columns = ['metric', 'mean', 'sample_sd']
    summary['n_seeds'] = len(frame)
    return summary


In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

collator = DataCollatorWithPadding(tokenizer, padding=True, pad_to_multiple_of=8)
baseline_batches_per_epoch = math.ceil(len(train_datasets['original']) / MICRO_BATCH)
baseline_steps_per_epoch = max(baseline_batches_per_epoch // GRAD_ACCUMULATION, 1)
matched_update_steps = int(EPOCHS * baseline_steps_per_epoch)
print({'matched_update_steps': matched_update_steps,
       'eval_every_steps': baseline_steps_per_epoch})

def optimizer_groups(model):
    groups = []
    for is_encoder, learning_rate in ((True, ENCODER_LR), (False, HEAD_LR)):
        for use_decay in (True, False):
            parameters = [parameter for name, parameter in model.named_parameters()
                if name.startswith('encoder.') == is_encoder and
                (parameter.ndim > 1) == use_decay]
            if parameters:
                groups.append({'params': parameters, 'lr': learning_rate,
                    'weight_decay': WEIGHT_DECAY if use_decay else 0.0})
    return groups

def collect_gate_diagnostics(model, dataset):
    loader = DataLoader(dataset, batch_size=EVAL_BATCH, shuffle=False,
        collate_fn=collator)
    gate_sum = gate_square_sum = gate_count = 0.0
    rho_value, layer_mixing = None, None
    model.eval()
    for batch in loader:
        batch.pop('labels', None)
        batch = {key: value.to(model.device) for key, value in batch.items()}
        gate, rho, mixing = model.gate_diagnostics(**batch)
        values = gate.detach().float().cpu()
        gate_sum += values.sum().item(); gate_square_sum += values.square().sum().item()
        gate_count += values.numel(); rho_value = float(rho.item())
        layer_mixing = [float(value) for value in mixing.detach().float().cpu()]
    mean = gate_sum / gate_count
    variance = max(0.0, gate_square_sum / gate_count - mean * mean)
    return {'rho': rho_value, 'gate_mean': mean, 'gate_std': variance ** 0.5,
        'effective_local_coefficient_mean': rho_value * mean,
        'last_layer_mixing_oldest_to_newest': layer_mixing}

def train_and_validate(variant):
    random.seed(SEED); np.random.seed(SEED); set_seed(SEED)
    variant_dir = RUN_DIR / variant
    variant_dir.mkdir(parents=True, exist_ok=False)
    model, parameter_counts = build_fresh_model()
    arguments = TrainingArguments(output_dir=str(variant_dir / 'checkpoints'),
        max_steps=matched_update_steps, learning_rate=ENCODER_LR,
        per_device_train_batch_size=MICRO_BATCH, per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUMULATION, optim='adamw_torch',
        weight_decay=WEIGHT_DECAY, lr_scheduler_type='linear', warmup_ratio=WARMUP_RATIO,
        max_grad_norm=1.0, eval_strategy='steps', save_strategy='steps',
        eval_steps=baseline_steps_per_epoch, save_steps=baseline_steps_per_epoch,
        load_best_model_at_end=True, metric_for_best_model='macro_f1',
        greater_is_better=True, save_total_limit=2, save_safetensors=True, fp16=True,
        seed=SEED, data_seed=SEED, report_to='none', logging_steps=50,
        dataloader_num_workers=0, remove_unused_columns=True, label_names=['labels'])
    optimizer = torch.optim.AdamW(optimizer_groups(model),
        betas=(0.9, 0.999), eps=1e-8)
    trainer = Trainer(model=model, args=arguments,
        train_dataset=train_datasets[variant], eval_dataset=validation_ds,
        processing_class=tokenizer, data_collator=collator,
        compute_metrics=compute_metrics, optimizers=(optimizer, None))
    print({'phase': 'train', 'seed': SEED, 'variant': variant,
        'train_rows': len(train_datasets[variant]), 'parameters': parameter_counts})
    trainer.train()
    best_model_dir = variant_dir / 'best_model'
    trainer.save_model(best_model_dir); tokenizer.save_pretrained(best_model_dir)
    output = trainer.predict(validation_ds)
    labels, logits = np.asarray(output.label_ids), np.asarray(output.predictions)
    raw_metrics = metrics_from_logits(labels, logits)
    calibration = fit_validation_biases(logits, labels,
        radius=CALIBRATION_RADIUS, step=CALIBRATION_STEP)
    biases = np.asarray(calibration['biases'])
    calibrated_metrics = metrics_from_logits(labels, logits + biases)
    probabilities = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()
    pd.DataFrame({'y_true': labels, 'y_pred_raw': logits.argmax(-1),
        'y_pred_calibrated': (logits + biases).argmax(-1),
        'logit_clean': logits[:, 0], 'logit_offensive': logits[:, 1],
        'logit_hate': logits[:, 2], 'p_clean': probabilities[:, 0],
        'p_offensive': probabilities[:, 1], 'p_hate': probabilities[:, 2]
    }).to_parquet(variant_dir / 'predictions_validation.parquet', index=False)
    diagnostics = collect_gate_diagnostics(trainer.model, validation_ds)
    result = {'experiment': 'OPTIMIZED_GATE_V4_AUGMENTATION_ABLATION',
        'variant': variant, 'seed': SEED, 'train_rows': len(train_datasets[variant]),
        'augmentation': 'none' if variant == 'original' else AUGMENTATION_VERSION,
        'augmentation_manifest': None if variant == 'original' else augmentation_manifest,
        'matched_update_steps': matched_update_steps,
        'decision_rules': {'primary': 'raw_argmax',
            'secondary': 'validation_only_class_bias'},
        'best_checkpoint': trainer.state.best_model_checkpoint,
        'best_model_dir': str(best_model_dir), 'parameters': parameter_counts,
        'raw_validation_metrics': raw_metrics, 'calibration': calibration,
        'calibrated_validation_metrics': calibrated_metrics,
        'gate_diagnostics': diagnostics, 'model_id': MODEL_ID,
        'model_revision': MODEL_SHA, 'dataset_revision': manifest['dataset_revision']}
    (variant_dir / 'validation_result.json').write_text(json.dumps(
        result, indent=2, ensure_ascii=False), encoding='utf-8')
    print(json.dumps({'phase': 'validation', 'variant': variant,
        'raw_metrics': raw_metrics, 'calibration': calibration,
        'calibrated_metrics': calibrated_metrics,
        'gate_diagnostics': diagnostics}, indent=2, ensure_ascii=False))
    del output, trainer, optimizer, model
    gc.collect(); torch.cuda.empty_cache()
    return result

# Hoàn tất train/validation cho cả hai variant trước khi đọc test.
validation_results = [train_and_validate(variant) for variant in MODEL_VARIANTS]
validation_comparison = pd.DataFrame([{
    'variant': result['variant'], 'train_rows': result['train_rows'],
    'raw_macro_f1': result['raw_validation_metrics']['macro_f1'],
    'raw_clean_fpr': clean_false_positive_rate(result['raw_validation_metrics']),
    'raw_offensive_f1': result['raw_validation_metrics']['per_class_f1']['OFFENSIVE'],
    'raw_hate_f1': result['raw_validation_metrics']['per_class_f1']['HATE'],
    'calibrated_macro_f1': result['calibrated_validation_metrics']['macro_f1'],
} for result in validation_results])
validation_comparison.to_csv(RUN_DIR / 'validation_comparison.csv', index=False)
display(validation_comparison)


## Tiêu chí đọc kết quả

Ưu tiên raw Macro-F1. Một kết quả đáng tiếp tục cần đồng thời: delta Macro-F1 dương, OFFENSIVE/HATE F1 không giảm, và CLEAN FPR không tăng quá mạnh. CI paired bootstrap chứa 0 nghĩa là một seed chưa đủ bằng chứng; khi đó nên xác nhận thêm seeds `52/62` thay vì tiếp tục nhìn test để chỉnh augmentation.

In [ ]:
def paired_bootstrap_macro_f1_delta(labels, baseline, augmented, seed):
    truth = np.asarray(labels)
    baseline, augmented = np.asarray(baseline), np.asarray(augmented)
    groups = [np.flatnonzero(truth == label) for label in range(3)]
    rng, deltas = np.random.default_rng(seed), []
    for _ in range(BOOTSTRAP_SAMPLES):
        indices = np.concatenate([rng.choice(group, len(group), replace=True)
            for group in groups])
        base_score = metrics_from_predictions(truth[indices], baseline[indices])['macro_f1']
        aug_score = metrics_from_predictions(truth[indices], augmented[indices])['macro_f1']
        deltas.append(aug_score - base_score)
    point = (metrics_from_predictions(truth, augmented)['macro_f1'] -
        metrics_from_predictions(truth, baseline)['macro_f1'])
    return {'delta_augmented_minus_original': float(point),
        'bootstrap_95ci': [float(value) for value in np.quantile(deltas, [0.025, 0.975])],
        'samples': BOOTSTRAP_SAMPLES, 'stratified_paired': True}

if not EVALUATE_TEST:
    print('TEST SKIPPED')
else:
    test_frame = load_verified_frame('test')
    test_ds = make_dataset(test_frame)
    test_results, prediction_cache = [], {}
    for variant_index, validation_result in enumerate(validation_results):
        variant = validation_result['variant']
        variant_dir = RUN_DIR / variant
        model = OptimizedGateViSoBERT.from_pretrained(
            validation_result['best_model_dir'], local_files_only=True)
        arguments = TrainingArguments(output_dir=str(variant_dir / 'test_trainer_tmp'),
            per_device_eval_batch_size=EVAL_BATCH, fp16=True, report_to='none',
            dataloader_num_workers=0)
        evaluator = Trainer(model=model, args=arguments, processing_class=tokenizer,
            data_collator=collator)
        output = evaluator.predict(test_ds)
        labels, logits = np.asarray(output.label_ids), np.asarray(output.predictions)
        biases = np.asarray(validation_result['calibration']['biases'])
        raw_prediction = logits.argmax(axis=-1)
        calibrated_prediction = (logits + biases).argmax(axis=-1)
        raw_metrics = metrics_from_predictions(labels, raw_prediction)
        calibrated_metrics = metrics_from_predictions(labels, calibrated_prediction)
        raw_metrics['macro_f1_bootstrap_95ci'] = bootstrap_macro_f1_ci(
            labels, raw_prediction, seed=SEED, seed_offset=2 * variant_index)
        calibrated_metrics['macro_f1_bootstrap_95ci'] = bootstrap_macro_f1_ci(
            labels, calibrated_prediction, seed=SEED, seed_offset=2 * variant_index + 1)
        probabilities = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()
        pd.DataFrame({'y_true': labels, 'y_pred_raw': raw_prediction,
            'y_pred_calibrated': calibrated_prediction,
            'logit_clean': logits[:, 0], 'logit_offensive': logits[:, 1],
            'logit_hate': logits[:, 2], 'p_clean': probabilities[:, 0],
            'p_offensive': probabilities[:, 1], 'p_hate': probabilities[:, 2]
        }).to_parquet(variant_dir / 'predictions_test.parquet', index=False)
        result = {**validation_result, 'raw_test_metrics': raw_metrics,
            'calibrated_test_metrics': calibrated_metrics,
            'test_bias_source': 'validation_only',
            'test_interpretation': TEST_INTERPRETATION}
        (variant_dir / 'test_result.json').write_text(json.dumps(
            result, indent=2, ensure_ascii=False), encoding='utf-8')
        print(json.dumps({'phase': 'test', 'variant': variant,
            'raw_metrics': raw_metrics, 'calibrated_metrics': calibrated_metrics,
            'frozen_validation_biases': biases.tolist()}, indent=2, ensure_ascii=False))
        test_results.append(result)
        prediction_cache[variant] = {'labels': labels, 'raw': raw_prediction,
            'calibrated': calibrated_prediction}
        del output, evaluator, model
        gc.collect(); torch.cuda.empty_cache()

    test_comparison = pd.DataFrame([{
        'variant': result['variant'], 'train_rows': result['train_rows'],
        'raw_macro_f1': result['raw_test_metrics']['macro_f1'],
        'raw_clean_fpr': clean_false_positive_rate(result['raw_test_metrics']),
        'raw_offensive_precision': result['raw_test_metrics']['per_class_precision']['OFFENSIVE'],
        'raw_offensive_recall': result['raw_test_metrics']['per_class_recall']['OFFENSIVE'],
        'raw_offensive_f1': result['raw_test_metrics']['per_class_f1']['OFFENSIVE'],
        'raw_hate_precision': result['raw_test_metrics']['per_class_precision']['HATE'],
        'raw_hate_recall': result['raw_test_metrics']['per_class_recall']['HATE'],
        'raw_hate_f1': result['raw_test_metrics']['per_class_f1']['HATE'],
        'calibrated_macro_f1': result['calibrated_test_metrics']['macro_f1'],
    } for result in test_results])
    test_comparison.to_csv(RUN_DIR / 'test_comparison.csv', index=False)
    display(test_comparison)

    original_cache = prediction_cache['original']
    augmented_cache = prediction_cache[AUGMENTATION_VERSION]
    paired = {
        'raw': paired_bootstrap_macro_f1_delta(original_cache['labels'],
            original_cache['raw'], augmented_cache['raw'], SEED + 9000),
        'calibrated': paired_bootstrap_macro_f1_delta(original_cache['labels'],
            original_cache['calibrated'], augmented_cache['calibrated'], SEED + 9001),
    }
    (RUN_DIR / 'paired_test_comparison.json').write_text(json.dumps(
        paired, indent=2, ensure_ascii=False), encoding='utf-8')
    summary = {'experiment': 'OPTIMIZED_GATE_V4_AUGMENTATION_ABLATION',
        'seed': SEED, 'variants': list(MODEL_VARIANTS),
        'augmentation_manifest': augmentation_manifest,
        'validation': validation_comparison.to_dict(orient='records'),
        'test': test_comparison.to_dict(orient='records'),
        'paired_test_macro_f1': paired,
        'test_interpretation': TEST_INTERPRETATION}
    (RUN_DIR / 'augmentation_ablation_summary.json').write_text(json.dumps(
        summary, indent=2, ensure_ascii=False), encoding='utf-8')
    print(json.dumps({'phase': 'test_comparison', 'paired_macro_f1': paired,
        'test_interpretation': TEST_INTERPRETATION}, indent=2, ensure_ascii=False))
    print(f'SUCCESS: {RUN_DIR}')
